<a href="https://colab.research.google.com/github/CalculatedContent/xgboost2ww/blob/main/notebooks/SpamBase_Hyperparameter_Sweep_Alpha2_ManifoldSearch_W1278910_Pareto_AllW.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# SpamBase Pareto Frontier (All W Matrices)

This notebook loads the saved results generated by `SpamBase_Hyperparameter_Sweep_Alpha2_ManifoldSearch_W1278910.ipynb`
and plots Pareto frontiers for each matrix kind (`W1`, `W2`, `W7`, `W8`, `W9`, `W10`).


In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

sns.set_theme(style="whitegrid")


In [ ]:
EXPERIMENT_NAME = "spambase_alpha2_manifoldsearch_w1278910"
RESULT_FILE = "spambase_aggregated_multiw.csv"

# Candidate roots for local and Colab usage.
candidate_roots = [
    Path("./results"),
    Path("../results"),
    Path("/content/drive/MyDrive/xgboost2ww_results"),
]

results_dir = None
for root in candidate_roots:
    candidate = (root / EXPERIMENT_NAME).resolve()
    if (candidate / RESULT_FILE).exists():
        results_dir = candidate
        break

if results_dir is None:
    looked = [str((root / EXPERIMENT_NAME).resolve()) for root in candidate_roots]
    raise FileNotFoundError(
        "Could not find saved results. Checked:\n- " + "\n- ".join(looked)
    )

result_path = results_dir / RESULT_FILE
print(f"Using results from: {result_path}")


In [ ]:
aggregated_multiw_df = pd.read_csv(result_path)
print("Shape:", aggregated_multiw_df.shape)
aggregated_multiw_df.head(3)


In [ ]:
def pareto_frontier_max_y(df, x_col, y_col):
    """
    Pareto frontier where smaller x is preferred and larger y is preferred.
    Keeps points that are not dominated by a prior point in x-sorted order.
    """
    sdf = df.sort_values([x_col, y_col], ascending=[True, False]).copy()
    running_best = float("-inf")
    keep_idx = []
    for idx, row in sdf.iterrows():
        y = row[y_col]
        if y >= running_best:
            keep_idx.append(idx)
            running_best = y
    frontier = sdf.loc[keep_idx].sort_values(x_col).copy()
    return frontier

w_order = ["W1", "W2", "W7", "W8", "W9", "W10"]
ws_in_data = [w for w in w_order if w in set(aggregated_multiw_df["matrix_kind"]) ]
print("Ws found:", ws_in_data)


In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(18, 10), sharex=False, sharey=True)
axes = axes.flatten()

for i, w in enumerate(w_order):
    ax = axes[i]
    wdf = aggregated_multiw_df[aggregated_multiw_df["matrix_kind"] == w].copy()

    if wdf.empty:
        ax.set_title(f"{w} (no data)")
        ax.axis("off")
        continue

    frontier = pareto_frontier_max_y(wdf, "alpha_primary_mean", "val_accuracy_mean")

    ax.scatter(
        wdf["alpha_primary_mean"],
        wdf["val_accuracy_mean"],
        s=18,
        alpha=0.35,
        color="#4C78A8",
        label="all configs",
    )
    ax.plot(
        frontier["alpha_primary_mean"],
        frontier["val_accuracy_mean"],
        color="#F58518",
        linewidth=2,
        marker="o",
        markersize=4,
        label="Pareto frontier",
    )

    ax.set_title(f"{w}: {len(frontier)} frontier points")
    ax.set_xlabel("alpha_primary_mean")
    if i % 3 == 0:
        ax.set_ylabel("val_accuracy_mean")
    ax.legend(loc="best", fontsize=8)

plt.suptitle("SpamBase Alpha vs Validation Accuracy Pareto Frontier (All W)", y=1.02, fontsize=14)
plt.tight_layout()
plt.show()


In [ ]:
# Optional: export figure for reports.
out_path = results_dir / "spambase_allW_pareto_frontiers.png"
fig = plt.gcf()
fig.savefig(out_path, dpi=180, bbox_inches="tight")
print(f"Saved figure to: {out_path}")
